# GOKO v2 — architecture POC

Runs the pipeline end to end on a folder of `.txt` notes and prints the payload at every
boundary. Each stage is its own cell that reads globals from the previous stage, so you can
stop anywhere and inspect what the next stage will receive.

**Run order matters.** Cells read state the previous cells built. Run them top to bottom.
Cell 19 (remap) rebuilds the state it owns from scratch on every run rather than appending
to it, so re-running is safe; if you edit a cell in the middle, re-run everything below it.
Cell numbers in the prose are the section numbers in these headings.

**Two modes.**

| | |
|---|---|
| `OFFLINE_MODE = False` | Calls your Azure deployment. Needs `settings.env`. |
| `OFFLINE_MODE = True`  | Replays recorded extractions from `fixtures/offline_extractions.json` for the bundled notes. No network, no key. A note with no recorded payload fails loudly. |

Offline mode exists so the *plumbing* can be verified without a deployment. It is never
selected silently: if there is no key and `OFFLINE_MODE` is `False`, cell 1 raises and tells
you which toggle to flip. Every artifact is stamped `model: "offline-replay"` so an offline
run can never be mistaken for a real one.

**What this POC does not implement.** The architecture trace describes more than this
notebook runs. Cell 24 lists the gaps explicitly so the run summary cannot be mistaken for a
complete implementation.


## 1 — Configuration

Edit this cell only. It prints which keys your `settings.env` actually contains (values
masked) so you can map them without opening the file.


In [ ]:
import os, json, re, sys
from pathlib import Path

# ---- EDIT THESE ------------------------------------------------------------
OFFLINE_MODE = True             # True = replay fixtures, no network. See cell 2.
ENV_FILE   = "settings.env"
NOTES_DIR  = "./notes"          # folder of {claim}_{note_id}.txt files
OUT_DIR    = "./poc_output"
FIXTURES   = "./fixtures/offline_extractions.json"
MODEL      = "gpt-luna-5.6"     # Azure *deployment name*, not the model family
API_VERSION_FALLBACK = "2024-10-21"
# ----------------------------------------------------------------------------

def load_env(path):
    found = {}
    p = Path(path)
    if not p.exists():
        print(f"!! {path} not found in {Path.cwd()}")
        return found
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip().strip('"').strip("'")
        found[k] = v
        os.environ.setdefault(k, v)
    return found

ENV = load_env(ENV_FILE)

def mask(v):
    if v is None: return None
    return v if len(v) < 12 else v[:6] + "..." + v[-4:]

print(f"keys found in {ENV_FILE}: {len(ENV)}")
for k in sorted(ENV):
    print(f"  {k:<40} = {mask(ENV[k])}")

# Best-effort mapping across the usual Azure naming variants.
def pick(*names):
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

AZURE_ENDPOINT = pick("AZURE_OPENAI_ENDPOINT", "AZURE_OAI_ENDPOINT",
                      "OPENAI_API_BASE", "AZURE_ENDPOINT")
AZURE_KEY      = pick("AZURE_OPENAI_API_KEY", "AZURE_OAI_KEY", "AZURE_OPENAI_KEY",
                      "OPENAI_API_KEY", "AZURE_API_KEY")
AZURE_VERSION  = pick("AZURE_OPENAI_API_VERSION", "OPENAI_API_VERSION",
                      "AZURE_API_VERSION") or API_VERSION_FALLBACK
DEPLOYMENT     = pick("AZURE_OPENAI_DEPLOYMENT", "AZURE_OAI_DEPLOYMENT",
                      "DEPLOYMENT_NAME") or MODEL

print("\nresolved:")
print(f"  endpoint   = {AZURE_ENDPOINT}")
print(f"  key        = {mask(AZURE_KEY)}")
print(f"  api_version= {AZURE_VERSION}")
print(f"  deployment = {DEPLOYMENT}")

if not OFFLINE_MODE and not (AZURE_ENDPOINT and AZURE_KEY):
    raise RuntimeError(
        "No Azure endpoint/key resolved and OFFLINE_MODE is False.\n"
        "Set them in settings.env (or assign AZURE_ENDPOINT / AZURE_KEY by hand in this "
        "cell), or set OFFLINE_MODE = True to replay the bundled fixtures.\n"
        "The lane does not silently fall back to fixtures: an offline run and a real run "
        "are different experiments and must not be confused in the output.")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"\nmode       = {'OFFLINE (replay)' if OFFLINE_MODE else 'LIVE (Azure)'}")


## 2 — Client

Uses the Azure OpenAI client, or the offline replay client. Both expose the same
`chat.completions.create(...)` surface, so no downstream cell knows which one it has.

`USE_STRUCTURED = False` falls back to prompt-instructed JSON plus validation — use it to
compare a deployment that does not support strict structured outputs.


In [ ]:
RUN_MODEL_TAG = "offline-replay" if OFFLINE_MODE else None

class OfflineReplayClient:
    """Replays recorded payloads. Raises for any note it has no recording for.

    The point of this class is that it cannot invent an extraction. A missing fixture is an
    error that lands in the run's failure accounting, not an empty result that looks like a
    note with nothing in it."""

    class _Completions:
        def __init__(self, outer): self.outer = outer
        def create(self, **kw):
            return self.outer._create(**kw)

    class _Chat:
        def __init__(self, outer): self.completions = OfflineReplayClient._Completions(outer)

    class _Usage:
        prompt_tokens = 0
        completion_tokens = 0
        def __repr__(self): return "usage(offline: 0/0)"

    def __init__(self, fixtures_path):
        self.chat = OfflineReplayClient._Chat(self)
        p = Path(fixtures_path)
        if not p.exists():
            raise FileNotFoundError(
                f"OFFLINE_MODE is on but {fixtures_path} is missing. It holds the recorded "
                f"payloads the replay client serves.")
        doc = json.loads(p.read_text())
        self.extractions = doc["extractions"]
        self.categories  = doc.get("categories", {})

    def _wrap(self, payload):
        usage = OfflineReplayClient._Usage()
        msg    = type("M", (), {"content": json.dumps(payload)})()
        choice = type("C", (), {"message": msg, "finish_reason": "stop"})()
        return type("R", (), {"choices": [choice], "usage": usage})()

    def _create(self, **kw):
        user = kw["messages"][-1]["content"]
        if "Reply with the single word" in user:
            msg    = type("M", (), {"content": "ready"})()
            choice = type("C", (), {"message": msg, "finish_reason": "stop"})()
            return type("R", (), {"choices": [choice],
                                  "usage": OfflineReplayClient._Usage()})()
        m = re.search(r"<note_id>(\d+)</note_id>", user)
        if m:
            nid = m.group(1)
            if nid not in self.extractions:
                raise RuntimeError(
                    f"offline_fixture_missing: no recorded extraction for note {nid}. "
                    f"Offline mode replays the bundled notes only — run with "
                    f"OFFLINE_MODE = False against a deployment to process new notes.")
            return self._wrap(self.extractions[nid])
        m = re.search(r'"entity_id":\s*"([^"]+)"', user)
        if m:
            eid = m.group(1)
            if eid not in self.categories:
                raise RuntimeError(
                    f"offline_fixture_missing: no recorded category for entity {eid}.")
            return self._wrap(self.categories[eid])
        raise RuntimeError("offline_fixture_missing: unrecognised request shape")

if OFFLINE_MODE:
    client = OfflineReplayClient(FIXTURES)
    DEPLOYMENT = RUN_MODEL_TAG
    print("!! OFFLINE REPLAY CLIENT — no network call will be made.")
    print("!! Every artifact this run writes is stamped model='offline-replay'.")
    print(f"   fixtures: {FIXTURES}  notes recorded: {sorted(client.extractions)}")
else:
    try:
        from openai import AzureOpenAI
    except ImportError:
        print("pip install openai")
        raise
    client = AzureOpenAI(
        azure_endpoint=AZURE_ENDPOINT,
        api_key=AZURE_KEY,
        api_version=AZURE_VERSION,
    )
    print(f"client ready -> {AZURE_ENDPOINT}")

USE_STRUCTURED = True     # flip to False to compare against prompt-only JSON
TEMPERATURE    = 0.0
MAX_TOKENS     = 4096     # extraction of a long note needs room; truncation is detected

print(f"deployment   -> {DEPLOYMENT}")
print(f"structured   -> {USE_STRUCTURED}")


## 3 — Smoke test

One trivial call. Fail here rather than 200 lines deep.


In [ ]:
r = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    temperature=TEMPERATURE,
    max_tokens=10,
)
print("response:", r.choices[0].message.content)
print("usage   :", r.usage)


## 4 — Extraction schema v0.1

Closed enums for entity and detail types. `action_type` and `subcategory` stay open — that
is the overflow channel you mine later to decide what gets promoted.

`lint_strict_schema` runs before the first call. Strict structured outputs reject a schema
that uses validation keywords like `minItems`, and the error arrives as a 400 at call time —
one per note, inside the extraction loop's exception handler. Checking the schema here turns
a whole run that silently extracts nothing into a failure on the cell that owns the mistake.


In [ ]:
SCHEMA_VERSION = "0.1"

ENTITY_TYPES = ["person", "organization", "vehicle"]
DETAIL_TYPES = ["address", "phone", "tin", "ssn", "npi", "bar_number", "vin"]
STANCES      = ["asserted", "denied", "disputed", "alleged"]
BASES        = ["stated", "inferred"]

EXTRACTION_SCHEMA = {
  "type": "object",
  "additionalProperties": False,
  "required": ["entity_mentions", "detail_mentions", "action_mentions"],
  "properties": {
    "entity_mentions": {
      "type": "array",
      "items": {
        "type": "object", "additionalProperties": False,
        "required": ["mention_id", "quote", "type", "occurrences"],
        "properties": {
          "mention_id": {"type": "string",
            "description": "Note-local id: m1, m2, m3."},
          "quote": {"type": "string",
            "description": "Verbatim phrase naming this party, copied character for "
                           "character. Include enough surrounding words that the phrase "
                           "appears only once in the note."},
          "type": {"type": "string", "enum": ENTITY_TYPES,
            "description": "Do not guess. If the party fits none of these, omit it."},
          "occurrences": {"type": "array", "items": {"type": "string"},
            "description": "Every other verbatim phrase in this note referring to the "
                           "same party, including pronouns and role references."},
        }}},
    "detail_mentions": {
      "type": "array",
      "items": {
        "type": "object", "additionalProperties": False,
        "required": ["detail_id","quote","raw_value","detail_type",
                     "issuer","owner_ref","basis"],
        "properties": {
          "detail_id": {"type": "string"},
          "quote": {"type": "string",
            "description": "Verbatim phrase containing the value plus enough "
                           "surrounding words to be unique in the note."},
          "raw_value": {"type": "string",
            "description": "The value alone, exactly as written. Do not reformat."},
          "detail_type": {"type": "string", "enum": DETAIL_TYPES},
          "issuer": {"type": ["string", "null"],
            "description": "Issuing state or jurisdiction where stated. Never infer."},
          "owner_ref": {"type": "string",
            "description": "mention_id of the owning party, or the literal string "
                           "UNASSIGNED. Proximity in the text is not ownership."},
          "basis": {"type": "string", "enum": BASES},
        }}},
    "action_mentions": {
      "type": "array",
      "items": {
        "type": "object", "additionalProperties": False,
        "required": ["action_id","quote","action_type","participants",
                     "time_qualifier","stance","basis"],
        "properties": {
          "action_id": {"type": "string"},
          "quote": {"type": "string"},
          "action_type": {"type": "string",
            "description": "Short verb phrase from the text, lowercase with "
                           "underscores, e.g. referred_to, treated_by."},
          "participants": {"type": "array",
            "description": "At least one participant. An action with no participant "
                           "cannot be attached to anything, so omit it instead.",
            "items": {"type": "object", "additionalProperties": False,
              "required": ["mention_id", "role"],
              "properties": {"mention_id": {"type": "string"},
                             "role": {"type": "string"}}}},
          "time_qualifier": {"type": ["string", "null"],
            "description": "As written. Do not resolve relative dates."},
          "stance": {"type": "string", "enum": STANCES},
          "basis": {"type": "string", "enum": BASES},
        }}},
  }}

# Keywords strict structured outputs reject. The API returns a 400 naming the keyword;
# catching it here instead means the failure names the cell that has to change.
STRICT_UNSUPPORTED = {"minItems", "maxItems", "minLength", "maxLength", "pattern",
                      "format", "minimum", "maximum", "default", "oneOf", "allOf", "not",
                      "uniqueItems", "multipleOf", "patternProperties"}

def lint_strict_schema(schema, path="$"):
    """Raises on anything strict mode will reject. Returns the count of objects checked."""
    n = 0
    if isinstance(schema, dict):
        bad = STRICT_UNSUPPORTED & set(schema)
        if bad:
            raise ValueError(
                f"strict schema violation at {path}: {sorted(bad)} not permitted under "
                f"strict structured outputs")
        if schema.get("type") == "object":
            n += 1
            if schema.get("additionalProperties") is not False:
                raise ValueError(f"strict schema violation at {path}: "
                                 f"additionalProperties must be false")
            props, req = set(schema.get("properties", {})), set(schema.get("required", []))
            if props != req:
                raise ValueError(
                    f"strict schema violation at {path}: every property must be required; "
                    f"missing from required: {sorted(props - req)}; "
                    f"required but absent: {sorted(req - props)}")
        for k, v in schema.items():
            n += lint_strict_schema(v, f"{path}.{k}")
    elif isinstance(schema, list):
        for i, v in enumerate(schema):
            n += lint_strict_schema(v, f"{path}[{i}]")
    return n

checked = lint_strict_schema(EXTRACTION_SCHEMA)
print(f"schema v{SCHEMA_VERSION}  (strict lint: {checked} object(s) OK)")
print(f"  entity types : {ENTITY_TYPES}")
print(f"  detail types : {DETAIL_TYPES}")
print(f"  action_type  : OPEN (overflow channel)")


## 5 — Claim id parser

`123456-789012-AB-01` → client, occurrence, coverage, sequence. Parsed strictly, never
sliced positionally — an off-format legacy id would otherwise produce a silently wrong
occurrence grouping.


In [ ]:
CLAIM_RE = re.compile(r"^(\d{6})-(\d{6})-([A-Za-z]{2})-(\d{2})$")
ID_FORMAT_VERSION = "1"

def parse_claim_id(raw):
    m = CLAIM_RE.match(raw.strip())
    if not m:
        return {"raw": raw, "valid": False, "flag": "malformed_claim_id"}
    client_id, occ, cov, seq = m.groups()
    return {
        "claim_id":       f"{client_id}-{occ}-{cov.upper()}-{seq}",
        "client_id":      client_id,
        "occurrence_id":  f"{client_id}-{occ}",
        "occurrence_seq": occ,
        "coverage_code":  cov.upper(),
        "claim_seq":      seq,
        "id_format_version": ID_FORMAT_VERSION,
        "valid": True,
    }

for t in ["123456-789012-AB-01", "123456-789012-ab-01", "LEGACY-4471902"]:
    print(f"{t:<24} -> {json.dumps(parse_claim_id(t))}")


## 6 — Load notes

Filename carries the claim id and note id. The filename pattern is anchored to the claim-id
format rather than a loose `(.+)_(\d{5,6})` — against a greedy prefix, `claim_1882130.txt`
matches with the note id silently truncated to `882130`. Malformed filenames are reported,
not skipped silently, and an empty load stops the run here instead of raising `IndexError`
two cells down.


In [ ]:
NOTE_FILE_RE = re.compile(
    r"^(?P<claim>\d{6}-\d{6}-[A-Za-z]{2}-\d{2})_(?P<note>\d+)\.txt$", re.IGNORECASE)

notes = []
bad_files = []
ndir = Path(NOTES_DIR)
if not ndir.exists():
    raise FileNotFoundError(f"{NOTES_DIR} does not exist (cwd={Path.cwd()})")

for f in sorted(ndir.glob("*.txt")):
    m = NOTE_FILE_RE.match(f.name)
    if not m:
        bad_files.append((f.name, "filename_pattern"))
        continue
    parsed = parse_claim_id(m.group("claim"))
    if not parsed.get("valid"):
        bad_files.append((f.name, "malformed_claim_id"))
        continue
    note_id = m.group("note")
    notes.append({
        "path": str(f),
        "note_id": int(note_id),
        "note_key": f"note:{note_id}",
        **{k: v for k, v in parsed.items() if k != "valid"},
        "raw_text": f.read_text(encoding="utf-8", errors="replace"),
    })

print(f"loaded {len(notes)} notes, {len(bad_files)} rejected")
for n in notes:
    print(f"  {n['note_key']:<14} claim={n['claim_id']}  occ={n['occurrence_id']}  "
          f"raw_len={len(n['raw_text'])}")
for f, why in bad_files:
    print(f"  REJECT {f}  ({why})")

dupes = {k for k in (n["note_key"] for n in notes)
         if [x["note_key"] for x in notes].count(k) > 1}
if dupes:
    raise RuntimeError(f"duplicate note ids across files: {sorted(dupes)} — note_key is a "
                       f"node key downstream and must be unique")

if not notes:
    raise RuntimeError(
        f"no usable notes in {NOTES_DIR}. Expected files named "
        f"{{claim_id}}_{{note_id}}.txt, e.g. 123456-789012-AB-01_188213.txt")

claims = sorted({n["claim_id"] for n in notes})
occurrences = sorted({n["occurrence_id"] for n in notes})
print(f"\n{len(claims)} claims across {len(occurrences)} occurrences")
for o in occurrences:
    cs = sorted({n['claim_id'] for n in notes if n['occurrence_id'] == o})
    print(f"  {o} -> {len(cs)} claim(s): {cs}")


## 7 — Cleaning with a non-destructive offset map

Every edit is recorded so a clean-text position can be translated back to a raw-text
position. Without this, every span in the finished dossier points at the wrong characters —
silently, and only in the investigator's viewer.

The cell ends with an exhaustive check: *every* clean index is translated and compared
against the raw text. A drift of one character anywhere fails here rather than in a
citation.


In [ ]:
CLEAN_POLICY = "clean-1.3"

def clean_with_map(raw):
    """Returns (clean_text, edits). edits let us map clean_idx -> raw_idx."""
    out, edits = [], []
    i, ci = 0, 0
    while i < len(raw):
        if raw.startswith("\r\n", i):
            out.append("\n"); edits.append({"raw_start": i, "raw_len": 2,
                                            "clean_start": ci, "clean_len": 1,
                                            "kind": "crlf_to_lf"})
            i += 2; ci += 1
        elif raw[i] == "\r":                      # lone CR, legacy Mac line ending
            out.append("\n"); i += 1; ci += 1     # length-preserving, no edit needed
        elif raw[i] in " \t":
            j = i
            while j < len(raw) and raw[j] in " \t":
                j += 1
            run = j - i
            out.append(" ")
            if run > 1:
                edits.append({"raw_start": i, "raw_len": run,
                              "clean_start": ci, "clean_len": 1,
                              "kind": "collapse_ws"})
            i = j; ci += 1
        else:
            out.append(raw[i]); i += 1; ci += 1
    return "".join(out), edits

def clean_to_raw(clean_idx, edits, clean_len=None):
    """Translate a cleaned-text index back to a raw-text index.

    `edits` is in ascending clean_start order, so the loop can stop at the first edit at or
    after the index. An edit that *starts* at clean_idx has not been passed yet: its drift
    belongs to positions after it, not to the index itself."""
    raw_idx = clean_idx
    for e in edits:
        if e["clean_start"] < clean_idx:
            raw_idx += (e["raw_len"] - e["clean_len"])
        else:
            break
    return raw_idx

for n in notes:
    n["clean_text"], n["edits"] = clean_with_map(n["raw_text"])
    n["clean_policy"] = CLEAN_POLICY

# Exhaustive offset-map check. Cheap at POC scale; the property it proves is the one every
# citation depends on.
def check_offset_map(note):
    raw, clean, edits = note["raw_text"], note["clean_text"], note["edits"]
    bad = []
    collapsed = set()
    for e in edits:
        for k in range(e["clean_start"], e["clean_start"] + e["clean_len"]):
            collapsed.add(k)
    for ci in range(len(clean)):
        ri = clean_to_raw(ci, edits, len(clean))
        if ri >= len(raw):
            bad.append((ci, ri, "past end of raw")); continue
        if ci in collapsed:
            continue                      # collapsed run: clean char is a substitution
        if clean[ci] != raw[ri]:
            bad.append((ci, ri, f"{clean[ci]!r} != {raw[ri]!r}"))
    end = clean_to_raw(len(clean), edits, len(clean))
    if end != len(raw):
        bad.append((len(clean), end, f"end maps to {end}, raw_len={len(raw)}"))
    return bad

offset_problems = 0
for n in notes:
    bad = check_offset_map(n)
    offset_problems += len(bad)
    print(f"{n['note_key']}  raw_len={len(n['raw_text'])}  clean_len={len(n['clean_text'])}"
          f"  edits={len(n['edits'])}  drift={len(n['raw_text']) - len(n['clean_text'])}"
          f"  map_check={'OK' if not bad else f'{len(bad)} BAD'}")
    for b in bad[:5]:
        print(f"    !! clean={b[0]} -> raw={b[1]}  {b[2]}")

if offset_problems:
    raise RuntimeError(f"{offset_problems} offset-map error(s). Every span downstream would "
                       f"point at the wrong characters.")

n0 = notes[0]
print("\nfirst 3 edits of", n0["note_key"])
for e in n0["edits"][:3]:
    print("   ", json.dumps(e))


## 8 — Lane A: pattern scan and checksums

High recall on structured identifiers, zero ownership. Checksums catch transposed digits
that look perfectly well-formed to a language model — the failure class the LLM lane cannot
see.

Two pattern changes from the naive version. The phone pattern now requires a separator or
parentheses, so it no longer competes with a bare ten-digit NPI; and a bare ten-digit run is
recorded with `cue: false`, because "ten digits with no NPI label" is a much weaker claim
than "ten digits after the word NPI" and the two should not be indistinguishable downstream.


In [ ]:
PATTERNS_VERSION = "1.5"

PATTERNS = {
  # cued NPI first: an explicit label is far stronger evidence than ten loose digits
  "npi_cued":   re.compile(r"\b(?:NPI|N\.P\.I\.)[\s#:]*(\d{10})\b", re.I),
  "npi":        re.compile(r"\b\d{10}\b"),
  "phone":      re.compile(r"(?:\(\d{3}\)\s?|\b\d{3}[-.\s])\d{3}[-.\s]?\d{4}\b"),
  "ssn":        re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
  "tin":        re.compile(r"\b\d{2}-\d{7}\b"),
  "vin":        re.compile(r"\b[A-HJ-NPR-Z0-9]{17}\b"),
  "bar_number": re.compile(r"\b(?:bar|ARDC)[\s#:]*(\d{6,8})\b", re.I),
  "address":    re.compile(r"\b\d{1,6}\s+[NSEW]?\.?\s?[A-Z][A-Za-z]+"
                           r"(?:\s+[A-Z][A-Za-z]+)*,?\s+[A-Z][a-z]+\s+[A-Z]{2}\s+\d{5}\b"),
}

# pattern name -> the detail_type it produces (several patterns can feed one type)
PATTERN_TYPE = {k: ("npi" if k.startswith("npi") else k) for k in PATTERNS}

def luhn_npi(v):
    """NPI check digit: prefix 80840, Luhn over the result."""
    if not (v.isdigit() and len(v) == 10):
        return "malformed"
    digits = [int(c) for c in ("80840" + v)]
    total, parity = 0, len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9: d -= 9
        total += d
    return "pass" if total % 10 == 0 else "FAIL_LUHN"

# Specific identifier patterns win over generic ones on the same characters.
PRIORITY = ["vin", "ssn", "tin", "bar_number", "npi_cued", "npi", "address", "phone"]

def pattern_scan(note):
    text, claimed, finds = note["clean_text"], [], []
    for pname in PRIORITY:
        rx = PATTERNS[pname]
        dtype = PATTERN_TYPE[pname]
        for m in rx.finditer(text):
            # When the pattern has a capture group, the SPAN is the group's span too —
            # otherwise a find labelled "bar_number 6224417" carries a span covering
            # "ARDC 6224417" and every citation built from it is off by five characters.
            s, e = m.span(1) if rx.groups else m.span(0)
            val   = m.group(1) if rx.groups else m.group(0)
            if any(s < ce and cs < e for cs, ce in claimed):
                continue                      # lower-priority overlap, suppress
            claimed.append((s, e))
            finds.append({"lane": "pattern", "pattern": pname, "detail_type": dtype,
                          "raw_value": val, "clean_span": [s, e],
                          "cue": pname.endswith("_cued"),
                          "checksum": luhn_npi(val) if dtype == "npi" else "n/a"})
    return sorted(finds, key=lambda f: f["clean_span"][0])

for n in notes:
    n["lane_pattern"] = pattern_scan(n)

for n in notes:
    print(f"\n{n['note_key']}  ({len(n['lane_pattern'])} finds)")
    for f in n["lane_pattern"]:
        chk = "" if f["checksum"] == "n/a" else f"  checksum={f['checksum']}"
        cue = "" if not f["cue"] else "  cued"
        print(f"  {f['detail_type']:<11} {f['raw_value']!r:<36} "
              f"span={f['clean_span']}{chk}{cue}")

fails = [f for n in notes for f in n["lane_pattern"] if f["checksum"] == "FAIL_LUHN"]
print(f"\nchecksum failures: {len(fails)}  <- data-quality signal, not extraction error")
uncued = [f for n in notes for f in n["lane_pattern"]
          if f["detail_type"] == "npi" and not f["cue"]]
print(f"uncued 10-digit runs read as npi: {len(uncued)}  <- weaker claim, see `cue` flag")


## 9 — Lane B: GLiNER (optional)

Zero-shot span detection for unpatterned names. Emits character spans natively, so these
finds skip quote resolution entirely. Gate this on measured LLM entity recall — if recall
against gold data is already adequate, the lane is infrastructure for nothing.

Disabled by default. When it *is* enabled its finds now reach reconciliation (cell 13) and
the mention pool, which is the whole point of running it; before, the lane computed spans
that nothing downstream ever read.


In [ ]:
GLINER_ENABLED = False          # set True after `pip install gliner`
GLINER_MODEL    = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.60
GLINER_INJECT   = False         # optional injection into the LLM prompt

gliner_model = None
if GLINER_ENABLED:
    try:
        from gliner import GLiNER
        gliner_model = GLiNER.from_pretrained(GLINER_MODEL)
        print(f"GLiNER loaded: {GLINER_MODEL}")
    except Exception as e:
        print(f"GLiNER unavailable ({e}); lane disabled")
        GLINER_ENABLED = False

def gliner_scan(note):
    if not GLINER_ENABLED or gliner_model is None:
        return []
    ents = gliner_model.predict_entities(note["clean_text"], ENTITY_TYPES, threshold=0.0)
    out = []
    for e in ents:
        out.append({"lane": "gliner", "label": e["label"], "text": e["text"],
                    "clean_span": [e["start"], e["end"]],
                    "score": round(float(e["score"]), 3),
                    "kept": float(e["score"]) >= GLINER_THRESHOLD})
    return out

for n in notes:
    n["lane_gliner"] = gliner_scan(n)

if GLINER_ENABLED:
    for n in notes:
        kept = [f for f in n["lane_gliner"] if f["kept"]]
        drop = [f for f in n["lane_gliner"] if not f["kept"]]
        print(f"\n{n['note_key']}  kept={len(kept)} discarded={len(drop)}")
        for f in kept:
            print(f"  KEEP {f['label']:<13} {f['text']!r:<32} "
                  f"span={f['clean_span']} score={f['score']}")
        for f in drop[:5]:
            print(f"  drop {f['label']:<13} {f['text']!r:<32} score={f['score']}")
    print("\ndiscard volume by label is how you tune the threshold against gold data")
else:
    print("Lane B disabled. Reconciliation will show detected_by without 'gliner'.")
    print("Injection path (GLINER_INJECT) destroys per-lane attribution — measure first.")


## 10 — Lane C: whole-note LLM extraction

The only per-note LLM call. Emits verbatim quotes, never offsets — the model is good at
copying text and bad at counting characters, so spans are computed by search in the next
cell.

A failed extraction is recorded as a failure, not as an empty note. The difference matters:
a 400 on the schema, a truncated response, or unparseable JSON all produce zero mentions,
and without the distinction the run summary reports a clean pipeline over an empty corpus.


In [ ]:
SYSTEM_PROMPT = """You extract structured entity intelligence from insurance claim notes.

Rules that matter more than completeness:
- Quote verbatim. Copy character for character from the note. Include enough surrounding
  words that each quote appears only ONCE in the note.
- Never emit character offsets or positions. Only quotes.
- One entity_mention per PARTY, not per occurrence. Put every other phrase referring to
  that same party (pronouns, role references) into `occurrences`.
- owner_ref is UNASSIGNED unless the text states or clearly implies ownership.
  Physical proximity in the text is NOT ownership.
- basis is "stated" when the text says it, "inferred" when you concluded it.
- stance describes how the text PRESENTS the event, not whether you believe it.
- action_type uses the text's own vocabulary, lowercase_with_underscores.
- Omit anything that does not fit the allowed types. Do not stretch a type to fit."""

EMPTY_EXTRACTION = {"entity_mentions": [], "detail_mentions": [], "action_mentions": []}

def build_messages(note):
    user = (f"Extract from this claim note.\n"
            f"<note_id>{note['note_id']}</note_id>\n\n<note>\n{note['clean_text']}\n</note>")
    if GLINER_ENABLED and GLINER_INJECT:
        cands = [f["text"] for f in note["lane_gliner"] if f["kept"]]
        if cands:
            user += ("\n\nCandidate entity spans detected by a separate model "
                     f"(verify each, do not trust blindly): {cands}")
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]

def extract_note(note):
    msgs = build_messages(note)
    kwargs = dict(model=DEPLOYMENT, messages=msgs, temperature=TEMPERATURE,
                  max_tokens=MAX_TOKENS)
    if USE_STRUCTURED:
        kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "note_extraction_v0_1",
                            "strict": True, "schema": EXTRACTION_SCHEMA}}
    else:
        kwargs["response_format"] = {"type": "json_object"}
        kwargs["messages"][0]["content"] += (
            "\n\nReturn ONLY JSON matching this schema:\n"
            + json.dumps(EXTRACTION_SCHEMA))
    r = client.chat.completions.create(**kwargs)
    choice = r.choices[0]
    if getattr(choice, "finish_reason", None) == "length":
        raise RuntimeError(
            f"response truncated at max_tokens={MAX_TOKENS}; the JSON is incomplete. "
            f"Raise MAX_TOKENS or split the note — do not treat this as an empty note.")
    payload = json.loads(choice.message.content)
    missing = [k for k in EMPTY_EXTRACTION if k not in payload]
    if missing:
        raise RuntimeError(f"extraction missing required keys: {missing}")
    return payload, r.usage

for n in notes:
    print(f"\n--- extracting {n['note_key']} ({len(n['clean_text'])} chars) ---")
    try:
        n["lane_llm"], usage = extract_note(n)
        n["extraction_error"] = None
        print(f"  usage: prompt={usage.prompt_tokens} completion={usage.completion_tokens}")
        print(f"  entities={len(n['lane_llm']['entity_mentions'])} "
              f"details={len(n['lane_llm']['detail_mentions'])} "
              f"actions={len(n['lane_llm']['action_mentions'])}")
        print(json.dumps(n["lane_llm"], indent=2)[:1200])
    except Exception as e:
        n["lane_llm"] = dict(EMPTY_EXTRACTION)
        n["extraction_error"] = f"{type(e).__name__}: {e}"
        print(f"  !! extraction FAILED: {n['extraction_error']}")

extraction_failures = [n for n in notes if n["extraction_error"]]
print(f"\nextraction failures: {len(extraction_failures)} / {len(notes)}")
if extraction_failures:
    print("!! These notes contributed nothing. Anything downstream is a partial run and")
    print("!! must not be read as a measurement. Failing notes:")
    for n in extraction_failures:
        print(f"   {n['note_key']}: {n['extraction_error']}")


## 11 — Quote resolution

Exact substring search, then normalized retry, then fuzzy. A multi-hit quote disambiguates
against the mention it claims a relationship to — and refuses to guess when no candidate is
clearly nearest.

Both retry paths return spans in the *cleaned text's* coordinates. The naive normalized
retry returns `normalized_text.index(...)`, which is an offset into a different string: for
any note where normalization changed a length before the match, the span silently points
somewhere else. The fuzzy path has the same problem in a different form — a window start is
not a match start. Both are aligned back to real coordinates here.


In [ ]:
from difflib import SequenceMatcher

FUZZY_MIN = 0.92
PROXIMITY_RATIO = 3.0

def _norm(s):
    return re.sub(r"\s+", " ", s).strip().lower()

def _norm_with_map(text):
    """_norm(text), plus norm_idx -> original_idx so a hit can be mapped back."""
    out, idx, i, n = [], [], 0, len(text)
    while i < n and text[i].isspace():
        i += 1
    while i < n:
        if text[i].isspace():
            j = i
            while j < n and text[j].isspace():
                j += 1
            if j < n:                     # trailing whitespace is dropped, like .strip()
                out.append(" "); idx.append(i)
            i = j
        else:
            out.append(text[i].lower()); idx.append(i); i += 1
    return "".join(out), idx

def _fuzzy_locate(quote, text):
    """Best approximate span for quote in text, aligned to the match, not to a window."""
    w = len(quote)
    if w == 0 or len(text) == 0:
        return None, 0.0
    step = max(1, w // 4)
    best_r, best_i = 0.0, None
    for i in range(0, max(1, len(text) - w + 1), step):
        r = SequenceMatcher(None, quote, text[i:i + w]).ratio()
        if r > best_r:
            best_r, best_i = r, i
    if best_i is None:
        return None, 0.0
    lo, hi = max(0, best_i - w), min(len(text), best_i + 2 * w)
    window = text[lo:hi]
    blocks = [b for b in SequenceMatcher(None, window, quote).get_matching_blocks()
              if b.size > 0]
    if not blocks:
        return None, best_r
    s = lo + blocks[0].a
    e = lo + blocks[-1].a + blocks[-1].size
    return [s, e], SequenceMatcher(None, quote, text[s:e]).ratio()

def resolve_quote(quote, text, anchor_pos=None):
    """-> (span_in_clean_text, method) or (None, reason)"""
    if not quote:
        return None, "empty_quote"
    hits = [m.start() for m in re.finditer(re.escape(quote), text)]
    if len(hits) == 1:
        return [hits[0], hits[0] + len(quote)], "exact"
    if len(hits) > 1:
        if anchor_pos is None:
            return None, "multi_hit_no_anchor"
        d = sorted((abs(h - anchor_pos), h) for h in hits)
        if len(d) > 1 and d[0][0] > 0 and d[1][0] / max(d[0][0], 1) < PROXIMITY_RATIO:
            return None, "multi_hit_ambiguous"
        h = d[0][1]
        return [h, h + len(quote)], "proximity"

    nq = _norm(quote)
    nt, nmap = _norm_with_map(text)
    nhits = [m.start() for m in re.finditer(re.escape(nq), nt)]
    if len(nhits) == 1 or (len(nhits) > 1 and anchor_pos is not None):
        if len(nhits) > 1:
            # rank normalized hits by their real position against the anchor
            cand = sorted((abs(nmap[h] - anchor_pos), h) for h in nhits)
            if len(cand) > 1 and cand[0][0] > 0 and \
               cand[1][0] / max(cand[0][0], 1) < PROXIMITY_RATIO:
                return None, "multi_hit_ambiguous"
            h = cand[0][1]
        else:
            h = nhits[0]
        s = nmap[h]
        e = nmap[h + len(nq) - 1] + 1      # map the LAST char, not h + len(nq)
        return [s, e], "normalized"

    span, ratio = _fuzzy_locate(quote, text)
    if span and ratio >= FUZZY_MIN:
        return span, f"fuzzy:{ratio:.2f}"
    return None, f"unverifiable_quote:{ratio:.2f}"

def resolve_note(note):
    text = note["clean_text"]
    resolved, methods, quotes, failed = {}, {}, {}, []

    def record(mid, kind, quote, anchor):
        span, how = resolve_quote(quote, text, anchor)
        quotes[mid] = quote
        if span:
            resolved[mid] = span
            methods[mid] = how
        else:
            failed.append({"id": mid, "kind": kind, "quote": quote, "reason": how})
        print(f"  {mid:<5} {kind:<7} {quote[:45]!r:<50} -> {span} {how}")

    for e in note["lane_llm"]["entity_mentions"]:
        record(e["mention_id"], "entity", e["quote"], None)
    for d in note["lane_llm"]["detail_mentions"]:
        anchor = resolved.get(d.get("owner_ref"), [None])[0]
        record(d["detail_id"], "detail", d["quote"], anchor)
    for a in note["lane_llm"]["action_mentions"]:
        pids = [p["mention_id"] for p in a.get("participants", [])]
        anchors = [resolved[p][0] for p in pids if p in resolved]
        anchor = sum(anchors) // len(anchors) if anchors else None
        record(a["action_id"], "action", a["quote"], anchor)
    return resolved, methods, quotes, failed

for n in notes:
    print(f"\n--- resolving {n['note_key']} ---")
    n["spans"], n["span_methods"], n["quotes"], n["quote_failures"] = resolve_note(n)
    print(f"  resolved={len(n['spans'])} failed={len(n['quote_failures'])}")


## 12 — Round-trip check

Map to raw coordinates, slice the original, compare **against the quote the model emitted**.

The comparison target is the whole point. Slicing raw and comparing it to the clean slice of
the same span re-derives one side from the other: it checks the offset map (already checked
exhaustively in cell 7) and passes any span, including a span pointing at the wrong sentence
entirely. Comparing the raw slice to the quote is what catches a hallucinated quote, a bad
fuzzy alignment, and a boundary bug.

Fuzzy-resolved spans are compared by ratio rather than equality — an approximate match
cannot be exact by definition — and are marked `approximate` so they stay visible.


In [ ]:
def round_trip(note):
    passes, fails = [], []
    for mid, span in note["spans"].items():
        rs  = clean_to_raw(span[0], note["edits"], len(note["clean_text"]))
        re_ = clean_to_raw(span[1], note["edits"], len(note["clean_text"]))
        raw_slice   = note["raw_text"][rs:re_]
        clean_slice = note["clean_text"][span[0]:span[1]]
        quote       = note["quotes"][mid]
        method      = note["span_methods"][mid]

        map_ok = _norm(raw_slice) == _norm(clean_slice)
        if method.startswith("fuzzy"):
            ratio    = SequenceMatcher(None, _norm(quote), _norm(raw_slice)).ratio()
            quote_ok = ratio >= FUZZY_MIN
            detail   = f"fuzzy ratio {ratio:.2f}"
        else:
            quote_ok = _norm(raw_slice) == _norm(quote)
            detail   = "exact-after-normalisation"
        ok = map_ok and quote_ok
        rec = {"id": mid, "clean_span": span, "raw_span": [rs, re_], "method": method,
               "raw_slice": raw_slice[:60], "map_ok": map_ok, "quote_ok": quote_ok,
               "approximate": method.startswith("fuzzy"), "ok": ok}
        (passes if ok else fails).append(rec)
        flag = "PASS" if ok else "FAIL"
        approx = "  (approximate)" if rec["approximate"] else ""
        print(f"  {flag} {mid:<5} clean={span} raw=[{rs}, {re_}]  "
              f"{raw_slice[:42]!r}{approx}")
        if not ok:
            print(f"       map_ok={map_ok} quote_ok={quote_ok} ({detail})")
            print(f"       quote was {quote[:60]!r}")
    return passes, fails

for n in notes:
    print(f"\n--- round-trip {n['note_key']} ---")
    n["rt_pass"], n["rt_fail"] = round_trip(n)
    n["raw_spans"] = {r["id"]: r["raw_span"] for r in n["rt_pass"]}
    # a span that fails the round trip is not a citation, so it is not carried forward
    for r in n["rt_fail"]:
        n["spans"].pop(r["id"], None)
    print(f"  pass={len(n['rt_pass'])} fail={len(n['rt_fail'])}")

total_fail = sum(len(n["rt_fail"]) for n in notes)
print(f"\nround-trip failures across all notes: {total_fail}")
print("Any failure here means a citation in the finished dossier would point at the wrong "
      "text, so the span is dropped and the mention goes to review.")


## 13 — Reconcile lanes

Match on normalized value **and overlapping span**, which is what makes the match a
statement about one occurrence of a value rather than about the value. Value-only matching
pairs the first pattern find with every LLM mention of the same value, so a note that says
the same phone number twice emits one reconciled detail and one phantom "recall gap".

Four outcomes: all lanes, pattern-only (the recall gap this lane exists for), GLiNER-only,
and LLM-only. GLiNER finds are reconciled against entity mentions, not details — the lane
produces spans for names, and names are not details.


In [ ]:
def normalize_detail(dtype, val):
    v = val.strip()
    if dtype == "phone":
        d = re.sub(r"\D", "", v)
        if len(d) == 10: return "+1" + d
        if len(d) == 11 and d.startswith("1"): return "+" + d
        return None
    if dtype in ("npi", "ssn", "tin", "bar_number"):
        d = re.sub(r"\D", "", v)
        return d or None
    if dtype == "vin":
        return v.upper()
    if dtype == "address":
        return re.sub(r"\s+", " ", re.sub(r"[.,]", "", v)).lower().replace(" ", "|")
    return v.lower()

def overlaps(a, b):
    return bool(a) and bool(b) and a[0] < b[1] and b[0] < a[1]

def reconcile(note):
    """LLM details x pattern finds. A pattern find is consumed by at most one mention."""
    out = []
    used_pattern = set()
    for d in note["lane_llm"]["detail_mentions"]:
        norm = normalize_detail(d["detail_type"], d["raw_value"])
        span = note["spans"].get(d["detail_id"])
        detected, checksum, cue = ["llm"], "n/a", None
        # Prefer a span-overlapping find; fall back to value-only when the LLM quote could
        # not be resolved (no span to compare) and exactly one find is still unclaimed.
        cands = [i for i, p in enumerate(note["lane_pattern"])
                 if i not in used_pattern
                 and p["detail_type"] == d["detail_type"]
                 and normalize_detail(p["detail_type"], p["raw_value"]) == norm]
        match = next((i for i in cands
                      if overlaps(span, note["lane_pattern"][i]["clean_span"])), None)
        if match is None and span is None and len(cands) == 1:
            match = cands[0]
        if match is not None:
            p = note["lane_pattern"][match]
            used_pattern.add(match)
            detected.append("pattern")
            checksum, cue = p["checksum"], p["cue"]
        out.append({"detail_type": d["detail_type"], "raw_value": d["raw_value"],
                    "normalized": norm, "clean_span": span,
                    "owner_ref": d["owner_ref"], "basis": d["basis"],
                    "issuer": d.get("issuer"), "detected_by": detected,
                    "checksum": checksum, "cue": cue, "recall_gap": False})
    for i, p in enumerate(note["lane_pattern"]):
        if i in used_pattern:
            continue
        out.append({"detail_type": p["detail_type"], "raw_value": p["raw_value"],
                    "normalized": normalize_detail(p["detail_type"], p["raw_value"]),
                    "clean_span": p["clean_span"], "owner_ref": "UNASSIGNED",
                    "basis": "stated", "issuer": None, "detected_by": ["pattern"],
                    "checksum": p["checksum"], "cue": p["cue"], "recall_gap": True})
    return out

def reconcile_entities(note):
    """GLiNER spans against LLM entity mentions. Returns (per-mention lanes, gliner-only)."""
    lanes = {e["mention_id"]: ["llm"] for e in note["lane_llm"]["entity_mentions"]}
    kept = [f for f in note["lane_gliner"] if f["kept"]]
    claimed = set()
    for e in note["lane_llm"]["entity_mentions"]:
        span = note["spans"].get(e["mention_id"])
        for i, f in enumerate(kept):
            if i in claimed:
                continue
            if overlaps(span, f["clean_span"]):
                lanes[e["mention_id"]].append("gliner")
                claimed.add(i)
                break
    gliner_only = [{"entity_candidate": True, "type": f["label"], "text": f["text"],
                    "clean_span": f["clean_span"], "score": f["score"],
                    "detected_by": ["gliner"], "loaded": False}
                   for i, f in enumerate(kept) if i not in claimed]
    return lanes, gliner_only

for n in notes:
    print(f"\n--- reconcile {n['note_key']} ---")
    n["details"] = reconcile(n)
    n["entity_lanes"], n["gliner_only"] = reconcile_entities(n)
    for d in n["details"]:
        tag = "  RECALL_GAP" if d["recall_gap"] else ""
        print(f"  {d['detail_type']:<11} {str(d['normalized'])[:34]:<36} "
              f"owner={d['owner_ref']:<12} by={'+'.join(d['detected_by'])}{tag}")
    for g in n["gliner_only"]:
        print(f"  entity_candidate {g['type']:<13} {g['text']!r} score={g['score']} "
              f"(gliner only, not loaded)")

gaps = [d for n in notes for d in n["details"] if d["recall_gap"]]
print(f"\nrecall gaps (pattern found, LLM missed): {len(gaps)}")
print("These would be lost silently without Lane A. They load as UNASSIGNED —")
print("a pattern match knows the value and nothing about who owns it.")


## 14 — Validation layer

Checks the schema could not express structurally: referential integrity, enum conformance,
format. Separate from the schema, and its failures mean different things.


In [ ]:
def validate_note(note):
    problems = []
    ids = {e["mention_id"] for e in note["lane_llm"]["entity_mentions"]}
    seen_mention_ids = [e["mention_id"] for e in note["lane_llm"]["entity_mentions"]]
    for mid in {m for m in seen_mention_ids if seen_mention_ids.count(m) > 1}:
        problems.append({"kind": "duplicate_mention_id", "value": mid})
    for e in note["lane_llm"]["entity_mentions"]:
        if e["type"] not in ENTITY_TYPES:
            problems.append({"kind": "enum_violation", "field": "entity.type",
                             "value": e["type"]})
    for d in note["details"]:
        if d["owner_ref"] not in ids and d["owner_ref"] != "UNASSIGNED":
            problems.append({"kind": "dangling_owner_ref", "value": d["owner_ref"]})
        if d["detail_type"] not in DETAIL_TYPES:
            problems.append({"kind": "enum_violation", "field": "detail_type",
                             "value": d["detail_type"]})
        if d["basis"] not in BASES:
            problems.append({"kind": "enum_violation", "field": "basis",
                             "value": d["basis"]})
        if d["normalized"] is None:
            problems.append({"kind": "unnormalizable", "value": d["raw_value"],
                             "detail_type": d["detail_type"]})
    for a in note["lane_llm"]["action_mentions"]:
        if not a.get("participants"):
            problems.append({"kind": "action_without_participant",
                             "value": a["action_id"]})
        for p in a.get("participants", []):
            if p["mention_id"] not in ids:
                problems.append({"kind": "dangling_participant",
                                 "action": a["action_id"], "value": p["mention_id"]})
        if a["stance"] not in STANCES:
            problems.append({"kind": "enum_violation", "field": "stance",
                             "value": a["stance"]})
    return problems

for n in notes:
    n["validation"] = validate_note(n)
    print(f"{n['note_key']}: {len(n['validation'])} problem(s)")
    for p in n["validation"]:
        print("   ", json.dumps(p))

print("\nenum_violation under structured output should be impossible.")
print("If it fires, the constraint was not in force — version skew or an")
print("unconstrained fallback path. That is a bug, not a data finding.")


## 15 — note_envelopes

Everything stamped with every version that could change the result.


In [ ]:
envelopes = []
for n in notes:
    env = {
      "note_id": n["note_id"], "note_key": n["note_key"],
      "claim_id": n["claim_id"], "client_id": n["client_id"],
      "occurrence_id": n["occurrence_id"], "coverage_code": n["coverage_code"],
      "versions": {"schema": SCHEMA_VERSION, "model": DEPLOYMENT,
                   "patterns": PATTERNS_VERSION,
                   "gliner": GLINER_MODEL if GLINER_ENABLED else None,
                   "clean_policy": CLEAN_POLICY, "chunk_policy": "whole_note",
                   "structured": USE_STRUCTURED, "offline": OFFLINE_MODE},
      "extraction_error": n["extraction_error"],
      "entity_mentions": [
        {**e, "clean_span": n["spans"].get(e["mention_id"]),
              "raw_span": n["raw_spans"].get(e["mention_id"]),
              "detected_by": n["entity_lanes"].get(e["mention_id"], ["llm"])}
        for e in n["lane_llm"]["entity_mentions"]],
      "entity_candidates": n["gliner_only"],
      "detail_mentions": n["details"],
      "action_mentions": [
        {**a, "clean_span": n["spans"].get(a["action_id"]),
              "raw_span": n["raw_spans"].get(a["action_id"])}
        for a in n["lane_llm"]["action_mentions"]],
      "review_items": (
        ([{"flag": "extraction_error", "detail": n["extraction_error"]}]
         if n["extraction_error"] else [])
        + [{"flag": f["reason"].split(":")[0], "id": f["id"], "quote": f["quote"]}
           for f in n["quote_failures"]]
        + [{"flag": "round_trip_fail", "id": r["id"], "method": r["method"]}
           for r in n["rt_fail"]]
        + [{"flag": "checksum_fail", "detail_type": d["detail_type"],
            "raw_value": d["raw_value"]}
           for d in n["details"] if d["checksum"] == "FAIL_LUHN"]
        + [{"flag": p["kind"], **p} for p in n["validation"]]),
    }
    envelopes.append(env)
    Path(OUT_DIR, f"envelope_{n['note_id']}.json").write_text(json.dumps(env, indent=2))

print(f"{len(envelopes)} envelopes written to {OUT_DIR}/")
for e in envelopes:
    print(f"  {e['note_key']:<14} entities={len(e['entity_mentions'])} "
          f"details={len(e['detail_mentions'])} actions={len(e['action_mentions'])} "
          f"review={len(e['review_items'])}")
print("\nsample:")
print(json.dumps(envelopes[0], indent=2)[:2000])


## 16 — mention_pool and blocking

Whole-note extraction means spans are already note-level. Nothing to rebase, no overlap to
deduplicate, and within-note coreference was the model's job. What remains is the hard part:
the same party across different notes.

Blocking keys are built per entity type. A person's discriminative token is the surname —
the last one. An organization's is the first: "Lakeshore Physical Therapy" and "Lakeshore
PT" share *Lakeshore* and nothing else, and a surname rule keys them on `therapy` and `pt`,
so the pair is never even proposed. The stopword list is split the same way, because `PT`
is a throwaway token in a person's name and the most informative token in a clinic's.


In [ ]:
from collections import defaultdict

PERSON_STOPWORDS = {"dr", "md", "do", "mr", "mrs", "ms", "esq", "jr", "sr", "the"}
ORG_STOPWORDS    = {"inc", "llc", "llp", "ltd", "corp", "co", "the", "and"}

def tokens(text):
    t = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    return [w for w in t.split() if len(w) > 1]

def name_keys(text, etype):
    stop = PERSON_STOPWORDS if etype == "person" else ORG_STOPWORDS
    toks = [w for w in tokens(text) if w not in stop]
    keys = set()
    if not toks:
        return keys
    keys.add(f"name_key:{'_'.join(toks)}")            # full normalized name
    if etype == "person":
        keys.add(f"name_key:{toks[-1]}")              # surname
        if len(toks) > 1:
            keys.add(f"name_key:{toks[-1]}|{toks[0][0]}")   # surname + first initial
    else:
        keys.add(f"name_key:{toks[0]}")               # leading distinctive token
        if len(toks) > 1:
            keys.add("name_key:" + "".join(w[0] for w in toks))  # acronym
    return keys

pools = {}
for claim in claims:
    pool = []
    for e in envelopes:
        if e["claim_id"] != claim: continue
        for m in e["entity_mentions"]:
            key = f"{e['note_key']}:{m['mention_id']}"
            details = [d for d in e["detail_mentions"]
                       if d["owner_ref"] == m["mention_id"] and d["normalized"]]
            blocks = name_keys(m["quote"], m["type"]) | {
                f"detail:{d['detail_type']}:{d['normalized']}" for d in details}
            pool.append({"key": key, "note_key": e["note_key"], "type": m["type"],
                         "quote": m["quote"], "occurrences": m.get("occurrences", []),
                         "details": [(d["detail_type"], d["normalized"]) for d in details],
                         "blocks": blocks})
    pools[claim] = pool

for claim, pool in pools.items():
    print(f"\n{claim}: {len(pool)} entity mentions")
    for p in pool:
        print(f"  {p['key']:<24} {p['type']:<13} {p['quote'][:32]!r}")
        print(f"      blocks: {sorted(p['blocks'])}")


## 17 — Pairwise scoring

Identifiers outvote names. A weak string match between two surface forms clears the high
band on the strength of a rare shared identifier — which is the weighting you want.

The two signals are scored independently and capped, rather than split as fractions of one
budget. Under `0.45·name + 0.55·shared`, a pair with no shared identifier tops out at 0.45,
so two mentions of the identical distinctive name in different notes can never reach `mid`,
let alone `high` — the band exists but no input reaches it. A shared identifier is weighted
by what that identifier is worth: an NPI has one owner, a street address has hundreds of
occupants, and a shared address is not evidence of identity in the way a shared NPI is.

Name similarity is `max(edit-distance ratio, token overlap)`. Raw string similarity punishes
a name for being written at a different length — "Dr. Monroe" against "A. Monroe, MD" is
0.70 on characters and 1.0 on tokens once titles are dropped, and the tokens are right.
Overlap divides by the *larger* token set, so a one-token name is not a perfect match for
every longer name that contains it.

`must_not_link` is a real constraint here, not a restatement of a low score. A band derived
from the same similarity number cannot veto anything: it says "these look unalike", which is
already what a low score says, and union-find will happily merge the pair transitively
anyway. A veto needs evidence of a different kind — a type mismatch, or two different values
of an identifier that a party can only have one of.

A shared address is deliberately weak. Two parties at one address reach `mid` at best —
flagged for a human, never merged automatically — because a building is not an identity.

**Known recall limit.** Abbreviations are not string similarity and not token containment:
`Lakeshore PT` against `Lakeshore Physical Therapy` scores `low` and stays two entities. The
pair is at least *proposed* now that organizations block on their leading token (cell 16),
so it is visible in the candidate list. Expanding an acronym token against a run of the
other name's tokens is the next thing to add here, and it needs gold data to justify its
threshold.


In [ ]:
BAND_HIGH, BAND_MID = 0.85, 0.60
SINGULAR_TYPES = ("npi", "ssn", "tin", "vin", "bar_number")

# How much a shared value of this type says about identity. Also used by cell 22 — one
# definition, so within-claim and cross-claim resolution cannot drift apart.
DETAIL_IDENTITY_WEIGHT = {
  "npi": 0.95, "ssn": 0.95, "vin": 0.90, "bar_number": 0.90, "tin": 0.85,
  "phone": 0.70, "address": 0.50,
}
NAME_WEIGHT, DETAIL_WEIGHT = 0.90, 0.65

def name_sim(a, b):
    return SequenceMatcher(None, _norm(a), _norm(b)).ratio()

def token_overlap(a, b, etype="person"):
    """Shared informative tokens over the larger token set.

    The denominator is `max`, not `min`: with `min`, any single-token name contained in a
    longer one scores 1.0, so "Dr. Monroe" would match "Dr. Smithson-Monroe" perfectly."""
    stop = PERSON_STOPWORDS if etype == "person" else ORG_STOPWORDS
    ta = {w for w in tokens(a) if w not in stop}
    tb = {w for w in tokens(b) if w not in stop}
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(len(ta), len(tb))

def name_match(a, b, etype="person"):
    """Character similarity or token overlap, whichever reads the pair more fairly."""
    return max(name_sim(a, b), token_overlap(a, b, etype))

def conflicting_singular(a, b):
    """Two different values of a detail type a party can only have one of."""
    for dt in SINGULAR_TYPES:
        va = {v for t, v in a["details"] if t == dt}
        vb = {v for t, v in b["details"] if t == dt}
        if va and vb and not (va & vb):
            return dt, sorted(va), sorted(vb)
    return None

def score_pair(a, b):
    shared = {x for x in a["blocks"] if x.startswith("detail:")} & \
             {x for x in b["blocks"] if x.startswith("detail:")}
    shared_w = max((DETAIL_IDENTITY_WEIGHT.get(x.split(":")[1], 0.5) for x in shared),
                   default=0.0)
    nm = name_match(a["quote"], b["quote"], a["type"])
    score = round(min(1.0, NAME_WEIGHT * nm + DETAIL_WEIGHT * shared_w), 3)
    veto = None
    if a["type"] != b["type"]:
        veto = f"type_mismatch:{a['type']}/{b['type']}"
    else:
        conflict = conflicting_singular(a, b)
        if conflict:
            veto = f"conflicting_{conflict[0]}:{conflict[1]}/{conflict[2]}"
    band = ("must_not_link" if veto else
            "high" if score >= BAND_HIGH else
            "mid"  if score >= BAND_MID else "low")
    return {"a": a["key"], "b": b["key"],
            "features": {"name_sim": round(name_sim(a["quote"], b["quote"]), 3),
                         "token_overlap": round(token_overlap(a["quote"], b["quote"],
                                                              a["type"]), 3),
                         "detail_agree": 1.0 if shared else None,
                         "detail_weight": shared_w,
                         "shared": sorted(shared), "type_match": a["type"] == b["type"]},
            "score": score, "band": band, "veto": veto}

pairs = {}
for claim, pool in pools.items():
    out = []
    for i, a in enumerate(pool):
        for b in pool[i+1:]:
            if a["note_key"] == b["note_key"]:   # within-note is the model's job
                continue
            if not (a["blocks"] & b["blocks"]):
                continue
            out.append(score_pair(a, b))
    pairs[claim] = out
    print(f"\n{claim}: {len(out)} candidate pair(s)")
    for p in out:
        print(f"  {p['a']} <-> {p['b']}")
        print(f"     {json.dumps(p['features'])}")
        print(f"     score={p['score']} band={p['band']}"
              + (f" veto={p['veto']}" if p['veto'] else ""))


## 18 — Union-find and id_map

Mid band writes no edge. Under-merge is recoverable and visible; over-merge is a false
attribution an investigator may act on.

Block constraints are enforced *against the component*, not against the pair. A
`must_not_link` pair is never a `high` pair, so filtering high edges by the blocked set can
never reject anything: A–B and B–C both merge, and A and C end up in one component with the
veto between them intact and ignored. Each candidate union is checked for any blocked pair
that would be brought together, and refused with a reason if one exists.

Entity numbering is derived from the lowest member key in each component rather than from
the union-find root, which depends on the order edges happened to arrive. `E1` has to mean
the same entity on the next run or the dossier is not comparable with itself.


In [ ]:
def union_find(pool, prs):
    parent = {p["key"]: p["key"] for p in pool}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def members(root):
        return [k for k in parent if find(k) == root]

    blocked = {tuple(sorted((p["a"], p["b"]))) for p in prs
               if p["band"] == "must_not_link"}
    refused = []

    for p in sorted(prs, key=lambda q: -q["score"]):
        if p["band"] == "must_not_link":
            print(f"  BLOCK {p['a']} XX {p['b']}  ({p['veto']})")
            continue
        if p["band"] == "mid":
            print(f"  FLAG  {p['a']} ?? {p['b']}  uncertain_merge (score {p['score']})")
            continue
        if p["band"] != "high":
            print(f"  none  {p['a']} .. {p['b']}  ({p['band']})")
            continue
        ra, rb = find(p["a"]), find(p["b"])
        if ra == rb:
            continue
        ma, mb = members(ra), members(rb)
        hit = next((pair for pair in blocked
                    if (pair[0] in ma and pair[1] in mb)
                    or (pair[0] in mb and pair[1] in ma)), None)
        if hit:
            refused.append({"edge": [p["a"], p["b"]], "would_merge": sorted(hit)})
            print(f"  REFUSE {p['a']} -- {p['b']}  (would merge blocked pair "
                  f"{hit[0]} / {hit[1]})")
            continue
        parent[ra] = rb
        print(f"  EDGE  {p['a']} -- {p['b']}  (score {p['score']})")

    comps = defaultdict(list)
    for k in parent:
        comps[find(k)].append(k)
    return comps, refused

id_maps, entity_sets, blocked_merges = {}, {}, {}
for claim, pool in pools.items():
    print(f"\n--- union-find {claim} ---")
    comps, refused = union_find(pool, pairs[claim])
    blocked_merges[claim] = refused
    idmap, ents = {}, []
    ordered = sorted(comps.values(), key=lambda ms: sorted(ms)[0])
    for i, membs in enumerate(ordered, start=1):
        eid = f"{claim}:E{i}"
        for m in membs:
            idmap[m] = eid
        src = [p for p in pool if p["key"] in membs]
        forms = []
        for s in src:
            forms.append(s["quote"]); forms.extend(s["occurrences"])
        uncertain = [p for p in pairs[claim] if p["band"] == "mid"
                     and (p["a"] in membs) != (p["b"] in membs)]
        ents.append({"entity_id": eid, "type": src[0]["type"],
                     "surface_forms": sorted(set(forms)),
                     "merge_provenance": {"merged_mentions": len(membs),
                                          "members": sorted(membs),
                                          "uncertain_merges": [
                                              {"with": p["b"] if p["a"] in membs
                                                       else p["a"],
                                               "score": p["score"]}
                                              for p in uncertain]}})
    id_maps[claim], entity_sets[claim] = idmap, ents
    print(f"  {len(pool)} mentions -> {len(ents)} entities")
    for k in sorted(idmap):
        print(f"    {k:<24} -> {idmap[k]}")


## 19 — Remap

The substitution where cross-note relationships materialize. Nothing detected them; identity
resolution did all of it.

The cell rebuilds each entity's details from the envelopes every time it runs rather than
appending to whatever was there. Appending makes the cell non-idempotent: a second run
doubles every detail, a third triples it, and nothing in the output says so — the dossier
just reports Dr. Monroe holding the same NPI three times.

The same value asserted in two notes is one detail with two pieces of evidence, not two
details, so identical `(type, value)` pairs are folded with their evidence kept.


In [ ]:
claim_unassigned, claim_actions, dangling_participants = {}, {}, {}

BASIS_RANK = {"stated": 2, "inferred": 1}

for claim in claims:
    idmap = id_maps[claim]
    ents = {e["entity_id"]: e for e in entity_sets[claim]}
    for e in ents.values():
        e["details"] = []          # rebuilt, not appended to — this cell is re-runnable
        e["flags"]   = []
    unassigned, actions, dangling = [], [], []

    for env in envelopes:
        if env["claim_id"] != claim: continue
        nk = env["note_key"]
        for d in env["detail_mentions"]:
            owner = idmap.get(f"{nk}:{d['owner_ref']}")
            rec = {k: d[k] for k in ("detail_type", "raw_value", "normalized", "basis",
                                     "detected_by", "checksum", "clean_span")}
            rec["evidence"] = [{"note_key": nk, "clean_span": d["clean_span"],
                                "raw_value": d["raw_value"]}]
            if owner:
                rec["owner_status"] = "assigned"
                ents[owner]["details"].append(rec)
            else:
                rec["owner_status"] = "unassigned"
                rec["note_key"] = nk
                unassigned.append(rec)
        for a in env["action_mentions"]:
            parts = []
            for p in a["participants"]:
                eid = idmap.get(f"{nk}:{p['mention_id']}")
                if eid is None:
                    dangling.append({"action": f"{nk}:{a['action_id']}",
                                     "mention_id": p["mention_id"]})
                parts.append({"entity_id": eid, "role": p["role"]})
            print(f"  {nk}:{a['action_id']}  {a['action_type']}")
            for p, q in zip(a["participants"], parts):
                mark = "" if q["entity_id"] else "   !! DANGLING"
                print(f"      {p['mention_id']} -> {q['entity_id']}  as {q['role']}{mark}")
            actions.append({"action_id": f"{nk}:{a['action_id']}",
                            "action_type": a["action_type"], "participants": parts,
                            "stance": a["stance"], "time_qualifier": a["time_qualifier"],
                            "quote": a["quote"], "raw_span": a.get("raw_span")})

    # fold identical (type, value) details; two notes saying the same thing is evidence,
    # not a second detail
    for e in ents.values():
        folded = {}
        for d in e["details"]:
            k = (d["detail_type"], d["normalized"])
            if k in folded:
                f = folded[k]
                f["detected_by"] = sorted(set(f["detected_by"]) | set(d["detected_by"]))
                f["evidence"].extend(d["evidence"])
                if BASIS_RANK.get(d["basis"], 0) > BASIS_RANK.get(f["basis"], 0):
                    f["basis"] = d["basis"]
                if f["checksum"] == "n/a":
                    f["checksum"] = d["checksum"]
            else:
                folded[k] = dict(d)
        e["details"] = list(folded.values())

    # contradictions on singular detail types
    for e in ents.values():
        for dt in SINGULAR_TYPES:
            vals = {d["normalized"] for d in e["details"] if d["detail_type"] == dt}
            if len(vals) > 1:
                e["flags"].append("contradictory_singular_detail")
                print(f"  !! {e['entity_id']} has {len(vals)} distinct {dt}: {vals}")
        if e["merge_provenance"]["uncertain_merges"]:
            e["flags"].append("uncertain_merge")

    entity_sets[claim] = list(ents.values())
    claim_unassigned[claim] = unassigned
    claim_actions[claim] = actions
    dangling_participants[claim] = dangling
    print(f"\n{claim}: {len(ents)} entities, {len(unassigned)} unassigned details, "
          f"{len(actions)} actions, {len(dangling)} dangling participant(s)")


## 20 — Category assignment

Six frozen values carry the score. `subcategory` is open and unscored — mined later for
promotion. The rule pre-pass removes the cases a model should never be asked about.

`insufficient_evidence` is in the *schema* enum even though it is not one of the six scored
categories. Constraining generation to the six and then treating "insufficient_evidence" as
a possible answer is a contradiction: the model cannot return a value the decoder forbids,
so every ambiguous party gets one of the six anyway, and the honest answer only exists as a
verdict the validator imposes after the fact. Giving the model the option is what makes the
refusal measurable.

The off-taxonomy fault is raised out of the loop rather than caught by it. An off-enum
return means the constraint was not in force — version skew, or an unconstrained fallback
path — and collapsing it into `insufficient_evidence` makes a pipeline bug look like a
recall problem in the evaluation.


In [ ]:
TAXONOMY_VERSION = "3"
CATEGORIES = ["medical", "legal", "repair_shop", "witness", "financier", "other"]
# what the model may return: the six scored values plus an explicit refusal
CATEGORY_ENUM = CATEGORIES + ["insufficient_evidence"]

def rule_prepass(entity):
    dts = {d["detail_type"] for d in entity["details"]}
    if entity["type"] == "person" and "npi" in dts:
        return "medical", "npi_present_on_person"
    if "bar_number" in dts:
        return "legal", "bar_number_present"
    if entity["type"] == "vehicle":
        return "other", "vehicle_is_not_a_party"
    return None, None

CAT_SCHEMA = {
  "type": "object", "additionalProperties": False,
  "required": ["category", "subcategory", "basis", "evidence_ids"],
  "properties": {
    "category": {"type": "string", "enum": CATEGORY_ENUM,
      "description": "Choose insufficient_evidence rather than guessing."},
    "subcategory": {"type": ["string", "null"],
      "description": "Free text, lowercase_with_underscores. Unscored."},
    "basis": {"type": "string", "enum": ["stated", "inferred"]},
    "evidence_ids": {"type": "array", "items": {"type": "string"},
      "description": "Only ids present in the packet you were given."},
  }}
lint_strict_schema(CAT_SCHEMA)

CAT_SYSTEM = (
    "Assign a category to this claim party. Choose only from the enum. If the evidence "
    "does not support any of the six categories, choose 'other'. If the evidence does not "
    "support a decision at all, choose 'insufficient_evidence' — that is a real answer, "
    "not a failure. Cite only evidence_ids you were given.")

class PipelineFault(RuntimeError):
    """A bug in the pipeline, not a finding about the data."""

def categorize_llm(packet):
    kwargs = dict(model=DEPLOYMENT, temperature=TEMPERATURE, max_tokens=512,
                  messages=[{"role": "system", "content": CAT_SYSTEM},
                            {"role": "user", "content": json.dumps(packet, indent=2)}])
    if USE_STRUCTURED:
        kwargs["response_format"] = {"type": "json_schema",
                                     "json_schema": {"name": "category_v3", "strict": True,
                                                     "schema": CAT_SCHEMA}}
    else:
        kwargs["response_format"] = {"type": "json_object"}
        kwargs["messages"][0]["content"] += (
            "\n\nReturn ONLY JSON matching this schema:\n" + json.dumps(CAT_SCHEMA))
    r = client.chat.completions.create(**kwargs)
    if getattr(r.choices[0], "finish_reason", None) == "length":
        raise RuntimeError("category response truncated at max_tokens")
    return json.loads(r.choices[0].message.content)

pipeline_faults = []

for claim in claims:
    print(f"\n--- categorizing {claim} ---")
    acts = claim_actions[claim]
    for e in entity_sets[claim]:
        cat, rule = rule_prepass(e)
        if cat:
            e["category"] = {"value": cat, "basis": "rule", "rule_id": rule,
                             "taxonomy_version": TAXONOMY_VERSION}
            e["subcategory"] = None
            print(f"  {e['entity_id']:<28} RULE  {rule} -> {cat}")
            continue
        mine = [a for a in acts
                if any(p["entity_id"] == e["entity_id"] for p in a["participants"])]
        ev = [{"evidence_id": a["action_id"], "quote": a["quote"]} for a in mine]
        packet = {"entity_id": e["entity_id"], "type": e["type"],
                  "surface_forms": e["surface_forms"],
                  "details": [{"detail_type": d["detail_type"],
                               "raw_value": d["raw_value"]} for d in e["details"]],
                  "actions": [{"action_type": a["action_type"],
                               "role": next(p["role"] for p in a["participants"]
                                            if p["entity_id"] == e["entity_id"])}
                              for a in mine],
                  "evidence_quotes": ev, "taxonomy_version": TAXONOMY_VERSION}
        print(f"  {e['entity_id']:<28} LLM   packet={json.dumps(packet)[:120]}...")
        try:
            res = categorize_llm(packet)
        except Exception as ex:
            print(f"      !! category call failed: {type(ex).__name__}: {ex}")
            e["category"] = {"value": "insufficient_evidence", "basis": "error",
                             "taxonomy_version": TAXONOMY_VERSION,
                             "flag": "category_call_failed", "error": str(ex)}
            e["subcategory"] = None
            continue

        if res.get("category") not in CATEGORY_ENUM:
            # not a finding: the constraint was not in force
            fault = {"error": "off_taxonomy_value", "returned": res.get("category"),
                     "entity_id": e["entity_id"],
                     "taxonomy_version_in_call": TAXONOMY_VERSION}
            pipeline_faults.append(fault)
            e["category"] = {"value": "PIPELINE_FAULT", "basis": "error",
                             "taxonomy_version": TAXONOMY_VERSION, **fault}
            e["subcategory"] = None
            print(f"      !! PIPELINE FAULT {json.dumps(fault)}")
            continue

        valid_ev = {x["evidence_id"] for x in ev}
        unresolved = [x for x in res.get("evidence_ids", []) if x not in valid_ev]
        if unresolved:
            print(f"      !! unresolved evidence citation {unresolved} "
                  f"-> insufficient_evidence")
            e["category"] = {"value": "insufficient_evidence", "basis": "llm",
                             "taxonomy_version": TAXONOMY_VERSION,
                             "flag": "unresolved_evidence_citation",
                             "unresolved": unresolved}
            e["subcategory"] = None
        else:
            e["category"] = {"value": res["category"], "basis": res["basis"],
                             "taxonomy_version": TAXONOMY_VERSION,
                             "evidence_ids": res.get("evidence_ids", [])}
            e["subcategory"] = res.get("subcategory")
            print(f"      -> {res['category']}  sub={res.get('subcategory')}  "
                  f"basis={res['basis']}")

if pipeline_faults:
    raise PipelineFault(
        f"{len(pipeline_faults)} off-taxonomy return(s): {json.dumps(pipeline_faults)}\n"
        f"The enum constraint was not in force. Check the taxonomy version in the call "
        f"against the validator, and that USE_STRUCTURED took effect. This is a bug to "
        f"fix, not a data quality finding to record.")


## 21 — Claim dossier

The frozen, scoreable artifact. Nothing enters the graph until this exists, which keeps your
evaluation independent of graph state.


In [ ]:
dossiers = {}
for claim in claims:
    p = parse_claim_id(claim)
    d = {"claim_id": claim, "client_id": p["client_id"],
         "occurrence_id": p["occurrence_id"], "coverage_code": p["coverage_code"],
         "versions": {"schema": SCHEMA_VERSION, "taxonomy": TAXONOMY_VERSION,
                      "model": DEPLOYMENT, "patterns": PATTERNS_VERSION,
                      "clean_policy": CLEAN_POLICY, "chunk_policy": "whole_note",
                      "offline": OFFLINE_MODE},
         "entities": entity_sets[claim],
         "unassigned_details": claim_unassigned[claim],
         "actions": claim_actions[claim],
         "blocked_merges": blocked_merges[claim],
         "dangling_participants": dangling_participants[claim],
         "review_items": [r for e in envelopes if e["claim_id"] == claim
                          for r in e["review_items"]]}
    dossiers[claim] = d
    Path(OUT_DIR, f"dossier_{claim}.json").write_text(json.dumps(d, indent=2))
    print(f"\n=== {claim} ===")
    print(f"  entities={len(d['entities'])} unassigned={len(d['unassigned_details'])} "
          f"actions={len(d['actions'])} review={len(d['review_items'])}")
    for e in d["entities"]:
        print(f"  {e['entity_id']:<28} {e['type']:<13} "
              f"{e['category']['value']:<22} sub={e.get('subcategory')}")
        print(f"      forms: {e['surface_forms']}")
        for dd in e["details"]:
            ev = f" x{len(dd['evidence'])}" if len(dd["evidence"]) > 1 else ""
            print(f"      {dd['detail_type']:<11} {dd['normalized']} "
                  f"({'+'.join(dd['detected_by'])}){ev}")
        if e["flags"]:
            print(f"      flags: {e['flags']}")
    for u in d["unassigned_details"]:
        print(f"  UNASSIGNED {u['detail_type']:<11} {u['normalized']}")

print(f"\ndossiers written to {OUT_DIR}/")


## 22 — Cross-claim resolution, occurrence-aware

The key move: identity and suspicion are separate scores from the same evidence. Two claims
in one occurrence are one incident — shared details are expected there and mean nothing as a
fraud signal, while meaning a great deal as evidence of identity.

Three changes from the naive version, all about the score being able to say what the bands
claim it can say.

1. **Rarity is `2/n`, capped at 1.** With `1/n`, a detail shared by exactly two entities —
   the minimum for a link to exist at all — scores 0.5, so no pair in any corpus can ever
   score above `0.45 + 0.55·0.5 = 0.7275`, and the `same` band at 0.85 is unreachable by
   construction. Unique-to-a-pair should be rarity 1.0.
2. **Distance is an additive prior, not a multiplier.** A multiplier on a base that already
   incorporates the evidence can only scale identity down, so "same occurrence" — the
   strongest identity context there is — cannot push a pair over a threshold it would
   otherwise miss.
3. **Detail type carries weight**, from the same table cell 17 uses. A shared NPI and a
   shared street address are not the same evidence: a building has hundreds of occupants,
   an NPI has one owner.

The weights below are placeholders. They are internally consistent — cell 23 proves each
band is reachable — but they are not calibrated against anything, and calibration needs gold
data.


In [ ]:
DISTANCE_PRIOR = {            # additive identity prior by relationship distance
  "same_occurrence":  0.25,
  "same_client":      0.10,
  "different_client": 0.00,
}
SUSPICION_WEIGHT = {          # the same evidence, read as a fraud signal
  "same_occurrence":  0.00,   # one incident, several coverages: expected
  "same_client":      0.25,
  "different_client": 1.00,
}
LINK_SAME, LINK_REVIEW = 0.85, 0.60
# DETAIL_IDENTITY_WEIGHT comes from cell 17 — the same table decides what a shared value
# is worth within a claim and across claims.

def distance(ca, cb):
    a, b = parse_claim_id(ca), parse_claim_id(cb)
    if a["occurrence_id"] == b["occurrence_id"]: return "same_occurrence"
    if a["client_id"] == b["client_id"]:         return "same_client"
    return "different_client"

# rarity: how many distinct entities across the corpus share this detail key
detail_index = defaultdict(set)
entity_index = {}
for claim, d in dossiers.items():
    for e in d["entities"]:
        entity_index[(claim, e["entity_id"])] = e
        for dd in e["details"]:
            if dd["normalized"]:
                detail_index[f"{dd['detail_type']}:{dd['normalized']}"].add(
                    (claim, e["entity_id"]))

def rarity(key):
    n = len(detail_index[key])
    if n < 2:
        return 0.0                      # cannot link anything
    return round(min(1.0, 2.0 / n), 3)  # 1.0 == unique to this pair

def best_name_sim(ea, eb):
    fa, fb = ea["surface_forms"] or [""], eb["surface_forms"] or [""]
    et = ea["type"] if ea["type"] == eb["type"] else "person"
    return max(name_match(x, y, et) for x in fa for y in fb)

print("shared detail keys across claims:")
for k in sorted(detail_index):
    owners = sorted(detail_index[k])
    cl = {c for c, _ in owners}
    if len(cl) > 1:
        print(f"  {k:<44} shared by {len(owners)} entities in {len(cl)} claims "
              f"rarity={rarity(k)}")

# one link per entity PAIR, carrying every detail they share — not one link per detail
pair_keys = defaultdict(list)
for k in sorted(detail_index):
    owners = sorted(detail_index[k])
    for i, a in enumerate(owners):
        for b in owners[i+1:]:
            if a[0] == b[0]:
                continue                # within-claim is cell 17's job
            pair_keys[(a, b)].append(k)

def score_link(a, b, keys):
    ca, ea_id = a
    cb, eb_id = b
    ea, eb = entity_index[a], entity_index[b]
    dist = distance(ca, cb)
    per_key = {k: round(DETAIL_IDENTITY_WEIGHT.get(k.split(":", 1)[0], 0.5) * rarity(k), 3)
               for k in keys}
    evidence = round(min(1.0, max(per_key.values())
                         + min(0.20, 0.10 * (len(keys) - 1))), 3)
    ns = best_name_sim(ea, eb)
    identity = round(min(1.0, DISTANCE_PRIOR[dist] + 0.55 * evidence + 0.25 * ns), 3)
    return {"entity_a": f"{ca}:{ea_id}".replace(f"{ca}:{ca}:", f"{ca}:"),
            "entity_b": f"{cb}:{eb_id}".replace(f"{cb}:{cb}:", f"{cb}:"),
            "distance": dist,
            "shared_details": keys, "shared_detail_count": len(keys),
            "detail_evidence": per_key, "evidence_strength": evidence,
            "features": {"name_sim": round(ns, 3),
                         "category_match": ea["category"]["value"] == eb["category"]["value"]},
            "identity_score": identity,
            "suspicion_score": round(SUSPICION_WEIGHT[dist] * evidence, 3),
            "outcome": ("same" if identity >= LINK_SAME else
                        "review" if identity >= LINK_REVIEW else "different"),
            "decided_by": None, "decided_at": None}

links = [score_link(a, b, keys) for (a, b), keys in sorted(pair_keys.items())]
for link in links:
    print(f"\n  {link['entity_a']}  <->  {link['entity_b']}")
    print(f"    distance={link['distance']}  shared={link['shared_details']}  "
          f"evidence={link['evidence_strength']}  name_sim={link['features']['name_sim']}")
    print(f"    identity={link['identity_score']} -> {link['outcome']}"
          f"   suspicion={link['suspicion_score']}")
    if link["distance"] == "same_occurrence":
        print("    (expected: one incident, multiple coverages — no fraud signal)")

Path(OUT_DIR, "identity_links.json").write_text(json.dumps(links, indent=2))
print(f"\n{len(links)} identity link(s) written")


## 23 — Self-tests

Invariants the pipeline is supposed to hold, checked against this run and against synthetic
inputs built to break them. Every one of these corresponds to a defect that a run can
otherwise complete cleanly while carrying: wrong spans that pass the round trip, a veto that
vetoes nothing, a scoring band no input can reach.

Run this cell before believing any number in the summary.


In [ ]:
_tests, _failures = [], []

def test(name):
    def deco(fn):
        _tests.append((name, fn)); return fn
    return deco

@test("offset map: every clean index maps to the matching raw character")
def _t():
    bad = [b for n in notes for b in check_offset_map(n)]
    assert not bad, f"{len(bad)} mismatched indices, first: {bad[:3]}"

@test("round trip: every retained span's raw slice matches the quote it came from")
def _t():
    for n in notes:
        for mid, span in n["spans"].items():
            rs  = clean_to_raw(span[0], n["edits"], len(n["clean_text"]))
            re_ = clean_to_raw(span[1], n["edits"], len(n["clean_text"]))
            raw_slice, quote = n["raw_text"][rs:re_], n["quotes"][mid]
            if n["span_methods"][mid].startswith("fuzzy"):
                r = SequenceMatcher(None, _norm(quote), _norm(raw_slice)).ratio()
                assert r >= FUZZY_MIN, f"{n['note_key']}:{mid} fuzzy ratio {r:.2f}"
            else:
                assert _norm(raw_slice) == _norm(quote), \
                    f"{n['note_key']}:{mid} -> {raw_slice!r} != {quote!r}"

@test("round trip rejects a span pointing at the wrong text")
def _t():
    n = notes[0]
    probe = {"spans": {"probe": [0, 10]}, "edits": n["edits"],
             "clean_text": n["clean_text"], "raw_text": n["raw_text"],
             "quotes": {"probe": "a quote that is definitely not at offset zero"},
             "span_methods": {"probe": "exact"}}
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        passes, fails = round_trip(probe)
    assert not passes and len(fails) == 1, "a wrong span was accepted"

@test("quote resolution: normalized retry returns clean-text coordinates")
def _t():
    text = "prefix\n\n\n\n\n\n\n\n\n\nthe target phrase here"
    span, how = resolve_quote("The Target  Phrase", text)
    assert how == "normalized", how
    assert text[span[0]:span[1]] == "the target phrase", repr(text[span[0]:span[1]])

@test("quote resolution: fuzzy retry aligns to the match, not to a window boundary")
def _t():
    text = "aaaa bbbb the claimant attended the appointment on friday morning zz tail"
    span, how = resolve_quote("the claimant attended the appointment on friday mornings",
                              text)
    assert how.startswith("fuzzy"), how
    got = text[span[0]:span[1]]
    assert got.startswith("the claimant") and got.endswith("morning"), repr(got)

@test("quote resolution: an unresolvable quote is refused, not placed")
def _t():
    span, how = resolve_quote("billed under NPI 9999999999",
                              "nothing in this text resembles that at all")
    assert span is None and how.startswith("unverifiable_quote"), (span, how)

@test("blocking: an org's leading token blocks, an abbreviation does not hide it")
def _t():
    a = name_keys("Lakeshore PT", "organization")
    b = name_keys("Lakeshore Physical Therapy", "organization")
    assert a & b, f"no shared block key: {sorted(a)} vs {sorted(b)}"

@test("must_not_link vetoes a transitive merge")
def _t():
    pool = [{"key": "n1:m1"}, {"key": "n2:m1"}, {"key": "n3:m1"}]
    prs = [{"a": "n1:m1", "b": "n2:m1", "band": "high", "score": 0.9, "veto": None},
           {"a": "n2:m1", "b": "n3:m1", "band": "high", "score": 0.9, "veto": None},
           {"a": "n1:m1", "b": "n3:m1", "band": "must_not_link", "score": 0.0,
            "veto": "conflicting_npi"}]
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        comps, refused = union_find(pool, prs)
    sizes = sorted(len(v) for v in comps.values())
    assert sizes == [1, 2], f"blocked pair was merged anyway: {dict(comps)}"
    assert refused, "no refusal recorded"

@test("pair scoring: every band is reachable, on names alone and on identifiers alone")
def _t():
    def mk(key, quote, details):
        return {"key": key, "quote": quote, "type": "organization", "details": details,
                "blocks": {f"detail:{t}:{v}" for t, v in details}}
    same_name = score_pair(mk("a", "Lakeshore Physical Therapy", []),
                           mk("b", "Lakeshore Physical Therapy", []))
    assert same_name["band"] == "high", f"identical names cannot merge: {same_name}"
    npi_only = score_pair(mk("a", "Alpha Clinic", [("npi", "1548392012")]),
                          mk("b", "Zeta Holdings", [("npi", "1548392012")]))
    assert npi_only["band"] in ("mid", "high"), f"a shared NPI decides nothing: {npi_only}"
    addr_only = score_pair(mk("a", "Alpha Clinic", [("address", "1100|w|lawrence")]),
                           mk("b", "Zeta Holdings", [("address", "1100|w|lawrence")]))
    assert addr_only["band"] != "high", \
        f"a shared address alone must never auto-merge two parties: {addr_only}"

@test("pair scoring: token overlap rescues a short form of the same name")
def _t():
    assert name_match("Dr. Monroe", "A. Monroe, MD", "person") == 1.0
    assert name_match("Northside Auto Body", "Midwest Spine Clinic", "organization") < 0.5

@test("pair scoring: a one-token name is not a perfect match for any name containing it")
def _t():
    assert token_overlap("Dr. Monroe", "Dr. Smithson-Monroe", "person") < 1.0

@test("must_not_link fires on two different values of a singular identifier")
def _t():
    a = {"key": "a", "quote": "Dr. Monroe", "type": "person",
         "details": [("npi", "1548392012")], "blocks": {"name_key:monroe"}}
    b = {"key": "b", "quote": "Dr. Monroe", "type": "person",
         "details": [("npi", "1999999999")], "blocks": {"name_key:monroe"}}
    r = score_pair(a, b)
    assert r["band"] == "must_not_link", r

@test("remap is idempotent: no entity carries a duplicated (type, value)")
def _t():
    for claim, es in entity_sets.items():
        for e in es:
            seen = [(d["detail_type"], d["normalized"]) for d in e["details"]]
            assert len(seen) == len(set(seen)), f"{e['entity_id']} duplicates: {seen}"

@test("entity ids are stable: numbering follows the lowest member key")
def _t():
    for claim, es in entity_sets.items():
        order = [sorted(e["merge_provenance"]["members"])[0] for e in es]
        assert order == sorted(order), f"{claim}: numbering not member-ordered: {order}"

@test("link scoring: every band is reachable")
def _t():
    rng = []
    for dist in DISTANCE_PRIOR:
        for ev in (0.0, 1.0):
            for ns in (0.0, 1.0):
                rng.append(min(1.0, DISTANCE_PRIOR[dist] + 0.55 * ev + 0.25 * ns))
    assert max(rng) >= LINK_SAME, f"'same' unreachable: max identity {max(rng):.3f}"
    assert min(rng) < LINK_REVIEW, f"'different' unreachable: min identity {min(rng):.3f}"
    mid = [x for x in rng if LINK_REVIEW <= x < LINK_SAME]
    assert mid, "'review' band unreachable"

@test("link scoring: a detail unique to one pair has rarity 1.0")
def _t():
    key = "npi:0000000001"
    detail_index[key] = {("c1", "E1"), ("c2", "E1")}
    try:
        assert rarity(key) == 1.0, rarity(key)
        detail_index[key] |= {("c3", "E1"), ("c4", "E1")}
        assert rarity(key) == 0.5, rarity(key)
    finally:
        del detail_index[key]

@test("both response schemas pass the strict-mode lint")
def _t():
    lint_strict_schema(EXTRACTION_SCHEMA)
    lint_strict_schema(CAT_SCHEMA)

@test("the category enum lets the model refuse")
def _t():
    assert "insufficient_evidence" in CAT_SCHEMA["properties"]["category"]["enum"]
    assert "insufficient_evidence" not in CATEGORIES, "refusal must not be a scored value"

@test("claim ids are parsed, not sliced")
def _t():
    assert parse_claim_id("LEGACY-4471902")["valid"] is False
    assert parse_claim_id("123456-789012-ab-01")["coverage_code"] == "AB"
    assert parse_claim_id("123456-789012-AB-01")["occurrence_id"] == "123456-789012"

@test("note filenames with an over-long note id are rejected, not truncated")
def _t():
    assert NOTE_FILE_RE.match("123456-789012-AB-01_188213.txt")
    m = NOTE_FILE_RE.match("123456-789012-AB-01_1882130.txt")
    assert m is None or m.group("note") == "1882130", "note id silently truncated"

@test("pattern lane: a captured value's span covers the value, not the cue")
def _t():
    probe = {"clean_text": "Attorney of record is J. Whitfield, ARDC 6224417."}
    finds = pattern_scan(probe)
    bar = [f for f in finds if f["detail_type"] == "bar_number"]
    assert bar, finds
    s, e = bar[0]["clean_span"]
    assert probe["clean_text"][s:e] == "6224417", repr(probe["clean_text"][s:e])

@test("pattern lane: a punctuated phone is not swallowed by the npi pattern")
def _t():
    probe = {"clean_text": "call (312) 555-0101 or NPI 1548392012 for records"}
    finds = {f["detail_type"]: f["raw_value"] for f in pattern_scan(probe)}
    assert finds.get("phone") == "(312) 555-0101", finds
    assert finds.get("npi") == "1548392012", finds

@test("checksum: a transposed NPI digit fails Luhn")
def _t():
    assert luhn_npi("1548392012") == "pass"
    assert luhn_npi("1548392018") == "FAIL_LUHN"
    assert luhn_npi("15483920") == "malformed"

@test("reconcile: the same value twice in one note is not a phantom recall gap")
def _t():
    probe = {"clean_text": "call (312) 555-0101 today, or (312) 555-0101 after five",
             "lane_llm": {"entity_mentions": [], "action_mentions": [], "detail_mentions": [
                 {"detail_id": "d1", "quote": "call (312) 555-0101 today",
                  "raw_value": "(312) 555-0101", "detail_type": "phone",
                  "issuer": None, "owner_ref": "UNASSIGNED", "basis": "stated"},
                 {"detail_id": "d2", "quote": "or (312) 555-0101 after five",
                  "raw_value": "(312) 555-0101", "detail_type": "phone",
                  "issuer": None, "owner_ref": "UNASSIGNED", "basis": "stated"}]},
             "lane_gliner": []}
    probe["lane_pattern"] = pattern_scan(probe)
    probe["spans"] = {d["detail_id"]: resolve_quote(d["quote"], probe["clean_text"])[0]
                      for d in probe["lane_llm"]["detail_mentions"]}
    out = reconcile(probe)
    assert len(out) == 2, [d["raw_value"] for d in out]
    assert not any(d["recall_gap"] for d in out), out

for name, fn in _tests:
    try:
        fn()
        print(f"  PASS  {name}")
    except Exception as ex:
        _failures.append((name, f"{type(ex).__name__}: {ex}"))
        print(f"  FAIL  {name}\n          {type(ex).__name__}: {ex}")

print(f"\n{len(_tests) - len(_failures)}/{len(_tests)} self-tests passed")
if _failures:
    raise AssertionError(f"{len(_failures)} self-test failure(s): "
                         + "; ".join(n for n, _ in _failures))


## 24 — What this notebook does not implement

The architecture trace describes more than this POC runs. Without this list a clean run
summary reads as a complete implementation.

| Architecture stage | State here |
|---|---|
| **UNASSIGNED retry** (2A) — one batched LLM call per claim that re-attempts ownership for unresolved details with full-claim context | **Absent.** Unresolved details stay `UNASSIGNED` in the dossier. The trace marks the retry optional in v1; the consequence is that a detail whose owner is only determinable from a *second* note is never attributed. |
| **Person projection** (2C) — canonical `Person` nodes recomputed from confirmed `IdentityLink`s | **Absent.** Links are scored and written; nothing projects them. |
| **Cluster-size alarm** (2C) — suspends a projection whose member count is implausible | **Absent**, and only meaningful once projection exists. It is the guard that stops a shared clinic line from chaining dozens of unrelated people into one Person. |
| **Watchlist matching** (2C) — match a Person against the watchlist, alert above threshold | **Absent.** No watchlist data exists yet. |
| **Graph load** (2C) — `MERGE` of Detail / Entity / Note / Evidence nodes into the store | **Absent.** The cross-claim query is done in memory over the dossiers, which is equivalent at this scale and not equivalent at archive scale. |
| **Oversized-note chunking** | **Absent.** `chunk_policy` is stamped `whole_note` everywhere. A note that exceeds the context window fails at extraction and is counted as a failure, which is the correct behaviour until the fallback exists. |
| **Rarity and threshold calibration** | Placeholders. Cell 23 proves the bands are reachable and self-consistent, which is not the same as correct. Calibration needs gold data. |

One structural point that no cell can carry: **within-note duplicate mentions are never
detected.** Assembly compares across notes only, on the principle that within-note
coreference is the model's job. When the model does emit two mentions for one party in one
note, nothing notices — they become two entities, and both survive into the dossier.


## 25 — Run summary

What to look at first: whether the run is valid at all, then review-queue volume, recall
gaps, and any round-trip failure.


In [ ]:
print("=" * 68)
print("RUN SUMMARY")
print("=" * 68)
print(f"mode                 : {'OFFLINE REPLAY (fixtures)' if OFFLINE_MODE else 'LIVE'}")
print(f"model / deployment   : {DEPLOYMENT}")
print(f"notes processed      : {len(notes)}")
print(f"claims               : {len(claims)}")
print(f"occurrences          : {len(occurrences)}")
print(f"entities resolved    : {sum(len(d['entities']) for d in dossiers.values())}")
print(f"actions              : {sum(len(d['actions']) for d in dossiers.values())}")
print(f"unassigned details   : {sum(len(d['unassigned_details']) for d in dossiers.values())}")
print(f"identity links       : {len(links)}")

ef = len(extraction_failures)
rt = sum(len(n["rt_fail"]) for n in notes)
qf = sum(len(n["quote_failures"]) for n in notes)
rg = len([d for n in notes for d in n["details"] if d["recall_gap"]])
ck = len([d for n in notes for d in n["details"] if d["checksum"] == "FAIL_LUHN"])
vp = sum(len(n["validation"]) for n in notes)
bm = sum(len(v) for v in blocked_merges.values())
dp = sum(len(v) for v in dangling_participants.values())
ie = len([e for d in dossiers.values() for e in d["entities"]
          if e["category"]["value"] == "insufficient_evidence"])

print("\nvalidity")
print(f"  extraction failures  : {ef}   <- non-zero means this run is PARTIAL")
if "_tests" in dir():
    print(f"  self-tests           : {len(_tests) - len(_failures)}/{len(_tests)} passed")
else:
    print( "  self-tests           : NOT RUN — run cell 23 before trusting anything here")

print("\nquality signals")
print(f"  round-trip failures  : {rt}   <- span dropped, mention sent to review")
print(f"  unresolved quotes    : {qf}   <- watch this; no overlap means no second chance")
print(f"  recall gaps (Lane A) : {rg}   <- what the LLM missed and patterns caught")
print(f"  checksum failures    : {ck}   <- data quality in the source notes")
print(f"  validation problems  : {vp}   <- enum violations here mean a pipeline bug")
print(f"  blocked merges       : {bm}   <- must_not_link refusals, with reasons")
print(f"  dangling participants: {dp}   <- action pointing at a mention that resolved to nothing")
print(f"  insufficient_evidence: {ie}   <- the model declining to guess; a real answer")

acts = [a["action_type"] for d in dossiers.values() for a in d["actions"]]
subs = [e.get("subcategory") for d in dossiers.values() for e in d["entities"]
        if e.get("subcategory")]
print("\noverflow channels — cluster these to decide what to promote")
print(f"  action_type values   : {sorted(set(acts))}")
print(f"  subcategory values   : {sorted(set(subs))}")

print()
if ef:
    print(f"!! {ef} of {len(notes)} notes produced nothing. Read nothing above as a")
    print("!! measurement: the corpus this run scored is not the corpus you loaded.")
elif OFFLINE_MODE:
    print("Offline replay: the plumbing ran end to end. No model was called, so none of")
    print("this measures extraction quality. Set OFFLINE_MODE = False for that.")
else:
    print("Run complete. See cell 24 for the architecture stages this POC does not cover.")
print(f"\nall artifacts in {OUT_DIR}/")
